# Model Analysis & Error Diagnostics

While standard evaluation metrics provide a high-level aggregate score, production-grade machine learning requires rigorous error analysis to understand *where*, *why*, and *how* a model fails. In this notebook, we load our serialized model and held-out test split to examine residual behavior, analyze feature attribution, inspect false positives/negatives in irrigation decisions, and evaluate performance trade-offs under real-world sensor variability.

In [15]:
from pathlib import Path
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

MODEL_PATH = PROJECT_ROOT / 'models' / 'irrigation_model.joblib'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
test_file = PROCESSED_DATA_DIR / 'test.csv'

TARGET_COLUMN = 'Irrigation_Need'
print("Paths initialized successfully!")
print("Project Root:", PROJECT_ROOT)
print("Model Path:", MODEL_PATH)
print("Processed Data Directory:", PROCESSED_DATA_DIR)

Paths initialized successfully!
Project Root: g:\AIOT\smart-irrigation
Model Path: g:\AIOT\smart-irrigation\models\irrigation_model.joblib
Processed Data Directory: g:\AIOT\smart-irrigation\data\processed


In [2]:

from pathlib import Path
import pandas as pd

# Define the absolute path to your processed data directory
PROCESSED_DATA_DIR = Path(r"G:\AIOT\smart-irrigation\data\processed")

# Retrieve and sort CSV files by modification time (newest first)
processed_files = sorted(
    PROCESSED_DATA_DIR.glob("*.csv"),
    key=lambda f: f.stat().st_mtime,
    reverse=True
)

# Check whether processed CSV files exist
if not processed_files:
    raise FileNotFoundError(
        f"No processed CSV files found in {PROCESSED_DATA_DIR}. Run notebook 02 first."
    )

# Load the latest generated dataset
latest_file = processed_files[0]

print(f"Loading latest processed data from: {latest_file.resolve()}")

data = pd.read_csv(latest_file)

# Display confirmation
print("\nDataset loaded successfully!")
print(f"Dataset shape: {data.shape}")
print("\nFirst five rows:")
display(data.head())


Loading latest processed data from: G:\AIOT\smart-irrigation\data\processed\irrigation_prediction_processed.csv

Dataset loaded successfully!
Dataset shape: (10000, 20)

First five rows:


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Rainfed,Reservoir,4.73,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Canal,Groundwater,12.22,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Drip,Reservoir,5.52,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Canal,Reservoir,1.43,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,Canal,River,2.52,No,60.86,South,Medium


In [5]:
from sklearn.model_selection import train_test_split
#Define the target column
TARGET_COLUMN ="Irrigation_Need"
#Separate impout features and target
X = data.drop(columns=[TARGET_COLUMN]).select_dtypes(
 include ="number"
)
y = data[TARGET_COLUMN]
#Recreating the original train-test split
X_train, X_test, y_train , y_test = train_test_split(
 X,
 y,
 test_size=0.2,
 random_state=42

)
#Display dataset information
print("Test dataset prepared successfully")

print(f"Training records:{len(X_train)}")
print(f"Testing records:{len(X_test)}")
print(f"Number of features:{X_test.shape[1]}")
print(f"Target variables:{TARGET_COLUMN}")

Test dataset prepared successfully
Training records:8000
Testing records:2000
Number of features:11
Target variables:Irrigation_Need


## 3. Generating model prediction
Use the trained Random Forest mdel to predict irrigation requirements from the test dataset.

In [1]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_PATH = PROJECT_ROOT / "models" / "irrigation_model.joblib"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TARGET_COLUMN = "Irrigation_Need"

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if "X_test" not in globals() or "y_test" not in globals():
    if "data" not in globals():
        processed_files = sorted(
            PROCESSED_DATA_DIR.glob("*.csv"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
        if not processed_files:
            raise FileNotFoundError(
                f"No processed CSV files found in {PROCESSED_DATA_DIR}"
            )
        data = pd.read_csv(processed_files[0])

    if TARGET_COLUMN not in data.columns:
        raise ValueError(f"Target column '{TARGET_COLUMN}' is missing from the data")
    X = data.drop(columns=[TARGET_COLUMN]).select_dtypes(include="number")
    y = data[TARGET_COLUMN]
    y_test = data[TARGET_COLUMN]
    _, X_test, _, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

model = joblib.load(MODEL_PATH)

print(f"Model loaded successfully: {type(model).__name__}")

y_pred = model.predict(X_test)
print("Model predictions generated successfully!")

comparison = pd.DataFrame(
    {"Actual": y_test, "Predicted": y_pred}, index=y_test.index
)
display(comparison.head())


Model loaded successfully: RandomForestClassifier
Model predictions generated successfully!


,Actual,Predicted
6252,Low,Medium
4684,Medium,Medium
1731,High,High
4742,Low,Low
4521,Low,Low
